### Coding base self attention mechanism class

In [4]:
import torch.nn as nn
import torch

class SelfAttentionV1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.Wq = nn.Parameter(torch.randn(d_in, d_out))
        self.Wk = nn.Parameter(torch.randn(d_in, d_out))
        self.Wv = nn.Parameter(torch.randn(d_in, d_out))

    def forward(self, x):
        queries = x @ self.Wq
        keys = x @ self.Wk
        values = x @ self.Wv

        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)
        context = attention_weights @ values
        return context

### Lets test with example inputs with context size of 6 and tokens of a sentence, with embedding dim as 3 and output dim as 2

In [6]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], 
   [0.55, 0.87, 0.66], 
   [0.57, 0.85, 0.64], 
   [0.22, 0.58, 0.33], 
   [0.77, 0.25, 0.10], 
   [0.05, 0.80, 0.55]] 
)

d_in = inputs.shape[1]
d_out = 2

torch.manual_seed(123)
attention_layer = SelfAttentionV1(d_in, d_out)
outputs = attention_layer(inputs)

print(outputs)


tensor([[0.2845, 0.4071],
        [0.2854, 0.4081],
        [0.2854, 0.4075],
        [0.2864, 0.3974],
        [0.2863, 0.3910],
        [0.2860, 0.4039]], grad_fn=<MmBackward0>)


### Making improvements on the V1 with linear layers and disable bias units since Linear layers have better weight initialization

In [9]:
class SelfAttentionV2(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.Wq = nn.Linear(d_in, d_out, bias=False)
        self.Wk = nn.Linear(d_in, d_out, bias=False)
        self.Wv = nn.Linear(d_in, d_out, bias=False)    

    def forward(self, x):
        queries = self.Wq(x)
        keys = self.Wk(x)
        values = self.Wv(x)

        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)
        context = attention_weights @ values
        return context 

torch.manual_seed(123)
attention_layer = SelfAttentionV2(d_in, d_out)
outputs = attention_layer(inputs)
print(outputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


### Lets move onto masked attention and before we implement the class, lets perform masking operations with the previous class outputs

In [11]:
queries = attention_layer.Wq(inputs)
keys = attention_layer.Wk(inputs)
attention_scores = queries @ keys.T

attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)
print(attention_weights)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


In [22]:
context_size = attention_scores.shape[0]
mask_matrix = torch.tril(torch.ones(context_size, context_size), diagonal=0)
print(mask_matrix)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [15]:
masked_attention = attention_weights * mask_matrix
print(masked_attention)

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)


### now if we go with softmax for ecah row, it would disrupt the probability distribution created by the earlier softmax

In [19]:
row_sums = masked_attention.sum(dim=-1, keepdim=True)
print(row_sums, "\n")

masked_attention_norm = masked_attention / row_sums
print(masked_attention_norm)

tensor([[0.1717],
        [0.3385],
        [0.5132],
        [0.6693],
        [0.8361],
        [1.0000]], grad_fn=<SumBackward1>) 

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)


### instead of this there is a more efficient method

In [26]:
mask = torch.triu(torch.ones(context_size, context_size), diagonal=1)
masked = attention_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)


In [27]:
attention_weights = torch.softmax(masked/keys.shape[-1]**0.5, dim=-1)
print(attention_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


### So les start coding the masked attention class

In [28]:
class MaskedAttentionV1(nn.Module):
    def __init__(self, d_in, d_out, context_size, dropout, bias=False):
        super().__init__()
        self.Wq = nn.Linear(d_in, d_out, bias=bias)
        self.Wk = nn.Linear(d_in, d_out, bias=bias)
        self.Wv = nn.Linear(d_in, d_out, bias=bias)    
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_size, context_size), diagonal=1))

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape
        queries = self.Wq(x)
        keys = self.Wk(x)       
        values = self.Wv(x)

        attention_scores = queries @ keys.transpose(1, 2)
        attention_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context = attention_weights @ values
        return context

### let us test it out with batched inputs

In [33]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape, "\n")

torch.manual_seed(123)

context_size = batch.shape[1]
masked_attention = MaskedAttentionV1(d_in, d_out, context_size, 0.3)

context = masked_attention(batch)

print(context)
print("context.shape:", context.shape)

torch.Size([2, 6, 3]) 

tensor([[[-0.6456,  0.3166],
         [-0.3120,  0.1530],
         [-0.9000, -0.0903],
         [-0.8107, -0.1204],
         [-0.7894, -0.1401],
         [-0.2140, -0.0731]],

        [[ 0.0000,  0.0000],
         [-0.8392,  0.0083],
         [-0.5523,  0.0052],
         [-0.6528, -0.1978],
         [-0.7894, -0.1401],
         [-0.6558, -0.1183]]], grad_fn=<UnsafeViewBackward0>)
context.shape: torch.Size([2, 6, 2])


### moving forward with multi headed attention

In [36]:
class MultiHeadAttentionV1(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_size, dropout, bias=False):
        super().__init__()
        self.heads = nn.ModuleList([
            MaskedAttentionV1(d_in, d_out, context_size, dropout, bias) for _ in range(num_heads)
        ])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

### Let us test it

In [37]:
torch.manual_seed(123)

context_size = batch.shape[1]
d_in, d_out = 3, 2
multiheadattention = MultiHeadAttentionV1(d_in, d_out, num_heads=2, context_size=context_size, dropout=0.0)

context = multiheadattention(batch)
print(context)
print("context.shape:", context.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context.shape: torch.Size([2, 6, 4])


### okay now that we understand how to write a simple wrapper, lets build teh whole class from scratch and make more modifications

In [38]:
class MultiHeadAttentionV2(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_size, dropout, bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.Wq = nn.Linear(d_in, d_out, bias=bias)
        self.Wk = nn.Linear(d_in, d_out, bias=bias)
        self.Wv = nn.Linear(d_in, d_out, bias=bias)
        self.projection = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("mask", torch.triu(torch.ones(context_size, context_size), diagonal=1))

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape
        
        queries = self.Wq(x).view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = self.Wk(x).view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.Wv(x).view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attention_scores = queries @ keys.transpose(2, 3)
        attention_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attention_weights = torch.softmax(attention_scores / self.head_dim**0.5, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context = attention_weights @ values
        context = context.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.d_out)
        context = self.projection(context)
        return context

### Lets test this now

In [39]:
torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
multiheadattention = MultiHeadAttentionV2(d_in, d_out, num_heads=2, context_size=context_length, dropout=0.0)

context_vecs = multiheadattention(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


### we have completed this module from scratch